In [0]:
import unittest
import json
from pyspark.sql import functions as F
from pyspark.testing.utils import assertDataFrameEqual, assertSchemaEqual
from py4j.protocol import Py4JJavaError
from pyspark.sql.utils import AnalysisException

# --- 1. CONFIGURATION & WIDGET PARSING ---
try:
    dbutils.widgets.text("datasets_json", '[]', "Datasets List (JSON Array)")
    raw_widget_data = dbutils.widgets.get("datasets_json")
    table_names = json.loads(raw_widget_data)
except json.JSONDecodeError as e:
    print(f"Error: Malformed JSON input in widget. Using empty list. Details: {e}")
    table_names = []
except Exception as e:
    print(f"Unexpected error during widget initialization: {e}")
    table_names = []

SCHEMA_PREFIX = "data_gold.gold"

class GoldDataQualityTests(unittest.TestCase):
    pk_mapping = {
        "dim_metric_dictionary": "metric_id",
        "dim_geography_gold": "unique_city_id",
        "fact_market_metrics_gold": "fact_market_metrics_id",
        "dim_county": "county_id",
        "dim_state": "state_id",
        "dim_region_type": "region_type_id"
    }

    @classmethod
    def setUpClass(cls):
        cls.spark = spark

    def load_table_safely(self, full_path):
        """Helper to catch missing tables before running assertions."""
        try:
            return self.spark.table(full_path)
        except AnalysisException as e:
            self.fail(f"Table Access Error: Could not find or access {full_path}. Check if the DLT pipeline succeeded. Details: {e}")
        except Exception as e:
            self.fail(f"Unexpected error loading table {full_path}: {e}")

    def test_basic_contract_and_integrity(self):
        if not table_names:
            self.skipTest("No tables provided via widget. Skipping basic integrity tests.")

        for table in table_names:
            full_path = f"{SCHEMA_PREFIX}.{table}"
            with self.subTest(table=full_path):
                # Safe loading with try-catch logic
                df = self.load_table_safely(full_path)
                
                try:
                    # --- TEST 1: Row Existence ---
                    row_count = df.count()
                    self.assertGreater(row_count, 0, f"Critical Error: {full_path} contains zero records.")

                    # --- TEST 2: Schema Contract ---
                    required_cols = ["load_dt", "source"]
                    for col in required_cols:
                        self.assertIn(col, df.columns, f"Mandatory column '{col}' missing in {full_path}")

                    # --- TEST 3: NULL Check on Primary Keys ---
                    pk_col = self.pk_mapping.get(table)
                    if not pk_col:
                        self.fail(f"Configuration Error: No Primary Key mapped for table {table}")
                    
                    null_count = df.filter(F.col(pk_col).isNull()).count()
                    self.assertEqual(null_count, 0, f"Integrity Failure: PK '{pk_col}' in {full_path} contains NULLs.")
                
                except Exception as e:
                    self.fail(f"Logic error during integrity test for {full_path}: {e}")

    def test_scd_type_2_metadata(self):
        for table in table_names:
            if table.startswith("dim_") and table != "dim_geography_gold":
                full_path = f"{SCHEMA_PREFIX}.{table}"
                with self.subTest(table=full_path):
                    df = self.load_table_safely(full_path)
                    try:
                        scd2_cols = ["start_dt", "end_dt", "is_active"]
                        for col in scd2_cols:
                            self.assertIn(col, df.columns, f"SCD2 Column '{col}' missing in {full_path}")
                    except Exception as e:
                        self.fail(f"Error checking SCD2 metadata for {full_path}: {e}")

    def test_intermediate_duplicate_check(self):
        for table in table_names:
            full_path = f"{SCHEMA_PREFIX}.{table}"
            with self.subTest(table=full_path):
                df = self.load_table_safely(full_path)
                try:
                    pk_col = self.pk_mapping.get(table)
                    if pk_col:
                        duplicate_count = df.groupBy(pk_col).count().filter("count > 1").count()
                        self.assertEqual(duplicate_count, 0, f"Uniqueness Error: PK '{pk_col}' has duplicates in {full_path}")
                except Exception as e:
                    self.fail(f"Error during duplicate check for {full_path}: {e}")

# --- 2. EXECUTION ---
if __name__ == "__main__":
    try:
        suite = unittest.TestLoader().loadTestsFromTestCase(GoldDataQualityTests)
        result = unittest.TextTestRunner(verbosity=2).run(suite)
        
        if not result.wasSuccessful():
            # This triggers the Databricks Job failure status
            raise Exception(f"Gold Table Quality Checks Failed. Failures: {len(result.failures)} | Errors: {len(result.errors)}")
    except Exception as e:
        print(f"Harness Execution Error: {e}")
        raise e